# HotpotQA Local OpenAI Paraphrase Generation

This notebook generates paraphrases locally with the OpenAI API. It is intentionally simple: read up to 200 source queries, generate 1 paraphrase per profile, run `natural_mild` first, then `natural_strong`, then `lexical_strong`. For 200 source queries this produces 600 paraphrase rows.

Set `OPENAI_API_KEY` in your environment before running the notebook. Outputs are written under `artifacts/hotpotqa_full/paraphrase/openai_generation/`.

`lexical_strong` is a stricter lexical-substitution profile: it should replace 2-3 non-entity content words while preserving the answerable meaning. If `artifacts/hotpotqa_full/paraphrase/validated/regeneration_needed.tsv` exists, the notebook switches to regeneration mode automatically. In that mode it reads only the missing source/profile rows and writes retry-safe outputs named `openai_paraphrase_regeneration_candidates.tsv` and `openai_paraphrase_regeneration_shortages.tsv` so the original 400-row candidate export is not overwritten.


In [1]:
# Run once if your local environment does not have these packages.
# !pip install -q openai pandas tqdm


In [2]:
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'evaluation').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root()
SOURCE_QUERY_LIMIT = 200
CANDIDATES_PER_QUERY = 1
GENERATION_PROFILES = ['natural_mild', 'natural_strong', 'lexical_strong']
SEQUENTIAL_BY_PROFILE = True
PROMPT_VERSION = 's4_openai_local_v2_lexical_strong'

RUN_DIR = REPO_ROOT / 'artifacts' / 'hotpotqa_full' / 'paraphrase' / 'openai_generation'
VALIDATED_DIR = REPO_ROOT / 'artifacts' / 'hotpotqa_full' / 'paraphrase' / 'validated'
REGENERATION_INPUT_TSV = VALIDATED_DIR / 'regeneration_needed.tsv'
REGENERATION_MODE = REGENERATION_INPUT_TSV.exists()
REGENERATION_RUN_TAG = os.environ.get('PARAPHRASE_REGENERATION_RUN_TAG', time.strftime('%Y%m%d%H%M%S'))
CHECKPOINT_DIR = RUN_DIR / 'paraphrase_checkpoints'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if REGENERATION_MODE:
    CANDIDATES_TSV = RUN_DIR / 'openai_paraphrase_regeneration_candidates.tsv'
    CANDIDATES_JSONL = RUN_DIR / 'openai_paraphrase_regeneration_candidates.jsonl'
    SHORTAGES_TSV = RUN_DIR / 'openai_paraphrase_regeneration_shortages.tsv'
    SHORTAGES_JSONL = RUN_DIR / 'openai_paraphrase_regeneration_shortages.jsonl'
    EXPECTED_INPUT_FILENAMES = ['regeneration_needed.tsv']
else:
    CANDIDATES_TSV = RUN_DIR / 'openai_paraphrase_candidates.tsv'
    CANDIDATES_JSONL = RUN_DIR / 'openai_paraphrase_candidates.jsonl'
    SHORTAGES_TSV = RUN_DIR / 'openai_paraphrase_shortages.tsv'
    SHORTAGES_JSONL = RUN_DIR / 'openai_paraphrase_shortages.jsonl'
    EXPECTED_INPUT_FILENAMES = [
        'hotpotqa_paraphrase_requests.tsv',
        'hotpotqa_full_dev_queries.tsv',
    ]

DOTENV_PATH = REPO_ROOT / '.env'

def load_repo_dotenv(path=DOTENV_PATH):
    if not path.exists():
        return False
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip(chr(34)).strip(chr(39))
        if key and key not in os.environ:
            os.environ[key] = value
    return True

load_repo_dotenv()
OPENAI_MODEL = os.environ.get('OPENAI_MODEL', 'combo')

if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in your shell or in a repo-root .env file before running this notebook.')
openai_base_url = os.environ.get('OPENAI_BASE_URL') or None
client = OpenAI(base_url=openai_base_url)

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_all_seeds(42)
print('openai_model:', OPENAI_MODEL)
print('openai_base_url:', openai_base_url or 'OpenAI default')
print('regeneration_mode:', REGENERATION_MODE)
print('regeneration_run_tag:', REGENERATION_RUN_TAG if REGENERATION_MODE else '(none)')
print('sequential_profiles:', GENERATION_PROFILES)
print('candidate_output:', CANDIDATES_TSV)


openai_model: combo
openai_base_url: http://localhost:20128/v1
regeneration_mode: True
sequential_profiles: ['natural_mild', 'natural_strong']
candidate_output: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_candidates.tsv


In [3]:
def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'evaluation').exists():
            return candidate
    return start


def list_visible_tsvs(root=None):
    repo_root = find_repo_root()
    roots = [
        Path.cwd(),
        repo_root,
        repo_root / 'evaluation' / 'results',
        repo_root / 'artifacts',
        repo_root / 'notebooks',
    ]
    if root is not None:
        roots.insert(0, Path(root))
    found = []
    for base in roots:
        if base.exists():
            found.extend(sorted(base.rglob('*.tsv')))
    return list(dict.fromkeys(found))


def classify_input_columns(columns):
    cols = set(columns)
    export_cols = {'source_query_id', 'original_query', 'support_doc_ids', 'qrels', 'paraphrase_profile'}
    base_cols = {'query_id', 'query', 'support_doc_ids'}
    if export_cols.issubset(cols):
        return 'paraphrase_export'
    if base_cols.issubset(cols):
        return 'base_queries'
    return 'unknown'


def discover_input_tsv():
    visible = list_visible_tsvs()
    preferred = []
    for name in EXPECTED_INPUT_FILENAMES:
        preferred.extend([path for path in visible if path.name == name])
    candidates = preferred + [path for path in visible if path not in preferred]
    checked = []
    for path in candidates:
        try:
            header = pd.read_csv(path, sep='\t', nrows=0)
        except Exception as exc:
            checked.append(f'{path}: unreadable: {exc}')
            continue
        schema = classify_input_columns(header.columns)
        checked.append(f'{path}: {schema}')
        if schema != 'unknown':
            return path, checked
    return None, checked


def build_qrels_json(row):
    qrels = str(row.get('qrels', '')).strip()
    if qrels:
        return qrels
    support_doc_ids = str(row.get('support_doc_ids', '')).strip()
    if not support_doc_ids:
        return '[]'
    doc_ids = [item.strip() for item in re.split(r'[,;|]', support_doc_ids) if item.strip()]
    return json.dumps(doc_ids, ensure_ascii=False)


def read_input_requests(path):
    header = pd.read_csv(path, sep='\t', nrows=0)
    schema_kind = classify_input_columns(header.columns)
    print('input_schema:', schema_kind)
    if schema_kind == 'paraphrase_export':
        usecols = ['source_query_id', 'original_query', 'support_doc_ids', 'qrels', 'paraphrase_profile', 'constraints']
        usecols = [col for col in usecols if col in header.columns]
        df = pd.read_csv(path, sep='\t', usecols=usecols, dtype=str, keep_default_na=False)
        return df[df['paraphrase_profile'].isin(GENERATION_PROFILES)].copy()
    if schema_kind == 'base_queries':
        usecols = ['query_id', 'query', 'support_doc_ids']
        df = pd.read_csv(path, sep='\t', usecols=usecols, dtype=str, keep_default_na=False, nrows=SOURCE_QUERY_LIMIT)
        if len(df) == SOURCE_QUERY_LIMIT:
            print('Limiting base query input from full file to SOURCE_QUERY_LIMIT rows.')
        rows = []
        for _, row in df.iterrows():
            for profile in GENERATION_PROFILES:
                rows.append({
                    'source_query_id': row['query_id'],
                    'original_query': row['query'],
                    'support_doc_ids': row.get('support_doc_ids', ''),
                    'qrels': build_qrels_json(row),
                    'paraphrase_profile': profile,
                    'constraints': 'Preserve named entities, numbers, dates, and the relation being asked. For lexical_strong, replace 2-3 non-entity content words with equivalent wording.',
                })
        expanded = pd.DataFrame(rows)
        print(f'Expanded base query input into {len(expanded)} profile requests.')
        return expanded
    raise ValueError(f'Unsupported input schema: {schema_kind}')

input_tsv, checked_inputs = discover_input_tsv()
if input_tsv is None:
    visible = '\n'.join(str(path) for path in list_visible_tsvs()) or '(none)'
    checked = '\n'.join(checked_inputs) or '(none)'
    raise FileNotFoundError(
        'Could not find a paraphrase export TSV with the required columns. '
        f'Visible TSVs:\n{visible}\nChecked schemas:\n{checked}'
    )

requests_df = read_input_requests(input_tsv)
requests_df['source_query_id'] = requests_df['source_query_id'].astype(str)
requests_df['original_query'] = requests_df['original_query'].astype(str)
requests_df['paraphrase_profile'] = requests_df['paraphrase_profile'].astype(str)
print('input_tsv:', input_tsv)
print('requests:', len(requests_df))
print(requests_df['paraphrase_profile'].value_counts().to_dict())


input_schema: paraphrase_export
input_tsv: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\validated\regeneration_needed.tsv
requests: 11
{'natural_mild': 10, 'natural_strong': 1}


In [4]:
def make_prompt(original_query, profile, constraints):
    if profile == 'natural_mild':
        style = 'Make a small natural wording change.'
        lexical_rule = 'Small syntax or wording changes are acceptable.'
    elif profile == 'lexical_strong':
        style = 'Rewrite with strong lexical substitution while preserving meaning.'
        lexical_rule = 'Replace 2-3 non-entity content words with equivalent terms or short phrases. Do not only reorder the original words. Keep named entities, numbers, dates, and answer relation unchanged.'
    else:
        style = 'Rewrite the question more substantially while preserving meaning.'
        lexical_rule = 'Change more than syntax when natural, but preserve the original answer relation.'
    return f'''Rewrite this HotpotQA question for retrieval robustness testing.
Return exactly one paraphrase.
Do not answer the question.
Do not add explanations.
Profile: {profile}
Style: {style}
Lexical rule: {lexical_rule}
Constraints: {constraints}
Keep named entities, numbers, dates, and the relation being asked unchanged.
Original question: {original_query}'''.strip()


def parse_candidate_list(text):
    cleaned = text.strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        parsed = cleaned
    if isinstance(parsed, dict):
        values = [parsed.get('paraphrase', '')]
    elif isinstance(parsed, list):
        values = parsed
    else:
        values = [str(parsed)]
    result = []
    seen = set()
    for value in values:
        text_value = str(value).strip().strip(chr(34))
        key = re.sub(r'\s+', ' ', text_value.lower())
        if text_value and key not in seen:
            seen.add(key)
            result.append(text_value)
        if len(result) >= CANDIDATES_PER_QUERY:
            break
    return result


def call_openai_with_retry(prompt, max_attempts=3):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {'role': 'system', 'content': 'You are a careful paraphrase generator for benchmark data. Preserve meaning exactly. Return JSON like {\"paraphrase\": \"...\"}.'},
                    {'role': 'user', 'content': prompt},
                ],
                temperature=0.4,
            )
            return response.choices[0].message.content or ''
        except Exception as exc:
            last_error = exc
            if attempt == max_attempts:
                raise
            time.sleep(2 ** attempt)
    raise last_error


def generate_candidates(original_query, profile, constraints):
    prompt = make_prompt(original_query, profile, constraints)
    raw_text = call_openai_with_retry(prompt)
    return parse_candidate_list(raw_text), raw_text


def write_artifacts(candidates, shortages):
    candidate_columns = [
        'variant_query_id', 'source_query_id', 'paraphrase_profile', 'candidate_index',
        'original_query', 'paraphrased_query', 'support_doc_ids', 'qrels',
        'model_id', 'prompt_version', 'generation_notes',
    ]
    shortage_columns = ['source_query_id', 'paraphrase_profile', 'requested_candidates', 'generated_candidates', 'error', 'raw_generation']
    candidates_df = pd.DataFrame(candidates, columns=candidate_columns)
    shortages_df = pd.DataFrame(shortages, columns=shortage_columns)
    candidates_df.to_csv(CANDIDATES_TSV, sep='\t', index=False)
    candidates_df.to_json(CANDIDATES_JSONL, orient='records', lines=True, force_ascii=False)
    shortages_df.to_csv(SHORTAGES_TSV, sep='\t', index=False)
    shortages_df.to_json(SHORTAGES_JSONL, orient='records', lines=True, force_ascii=False)
    print('wrote:', CANDIDATES_TSV, len(candidates_df))
    print('wrote:', SHORTAGES_TSV, len(shortages_df))


In [5]:
all_candidates = []
all_shortages = []
for profile in GENERATION_PROFILES:
    print(f'Generating profile: {profile}')
    profile_df = requests_df[requests_df['paraphrase_profile'] == profile].copy()
    profile_candidates = []
    profile_shortages = []
    for _, row in tqdm(profile_df.iterrows(), total=len(profile_df), desc=profile):
        source_query_id = str(row['source_query_id'])
        original_query = str(row['original_query'])
        constraints = str(row.get('constraints', 'Preserve named entities, numbers, dates, and relation meaning.'))
        started = time.time()
        try:
            candidates, raw_text = generate_candidates(original_query, profile, constraints)
            error = ''
        except Exception as exc:
            candidates, raw_text, error = [], '', repr(exc)
        for idx, paraphrased_query in enumerate(candidates, start=1):
            variant_query_id = f'{source_query_id}__{profile}__regen{REGENERATION_RUN_TAG}_{idx}' if REGENERATION_MODE else f'{source_query_id}__{profile}__{idx}'
            profile_candidates.append({
                'variant_query_id': variant_query_id,
                'source_query_id': source_query_id,
                'paraphrase_profile': profile,
                'candidate_index': idx,
                'original_query': original_query,
                'paraphrased_query': paraphrased_query,
                'support_doc_ids': row.get('support_doc_ids', ''),
                'qrels': row.get('qrels', ''),
                'model_id': OPENAI_MODEL,
                'prompt_version': PROMPT_VERSION,
                'generation_notes': f'local_openai_sequential; regeneration_mode={REGENERATION_MODE}; elapsed_seconds={time.time() - started:.2f}',
            })
        if len(candidates) < CANDIDATES_PER_QUERY or error:
            profile_shortages.append({
                'source_query_id': source_query_id,
                'paraphrase_profile': profile,
                'requested_candidates': CANDIDATES_PER_QUERY,
                'generated_candidates': len(candidates),
                'error': error,
                'raw_generation': raw_text,
            })
        if len(profile_candidates) % 25 == 0:
            pd.DataFrame(profile_candidates).to_csv(CHECKPOINT_DIR / f'{profile}_candidates.tsv', sep='	', index=False)
            pd.DataFrame(profile_shortages).to_csv(CHECKPOINT_DIR / f'{profile}_shortages.tsv', sep='	', index=False)
    all_candidates.extend(profile_candidates)
    all_shortages.extend(profile_shortages)
    pd.DataFrame(profile_candidates).to_csv(CHECKPOINT_DIR / f'{profile}_candidates.tsv', sep='	', index=False)
    pd.DataFrame(profile_shortages).to_csv(CHECKPOINT_DIR / f'{profile}_shortages.tsv', sep='	', index=False)
    write_artifacts(all_candidates, all_shortages)
    if profile == 'natural_mild':
        print('Finished natural_mild before natural_strong starts.')

write_artifacts(all_candidates, all_shortages)
print('complete')


Generating profile: natural_mild


natural_mild:   0%|          | 0/10 [00:00<?, ?it/s]

wrote: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_candidates.tsv 10
wrote: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_shortages.tsv 0
Finished natural_mild before natural_strong starts.
Generating profile: natural_strong


natural_strong:   0%|          | 0/1 [00:00<?, ?it/s]

wrote: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_candidates.tsv 11
wrote: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_shortages.tsv 0
wrote: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_candidates.tsv 11
wrote: D:\VSCODE\vdt-meeting-search\artifacts\hotpotqa_full\paraphrase\openai_generation\openai_paraphrase_regeneration_shortages.tsv 0
complete


## Output files

The main files are `openai_paraphrase_candidates.tsv`, `openai_paraphrase_candidates.jsonl`, `openai_paraphrase_shortages.tsv`, and `openai_paraphrase_shortages.jsonl`. Local validation still decides which rows are accepted into `mild_200`, `strong_200`, and `lexical_strong_200`.
